# Project 8: Long-Term Memory Agent

Compare recent-window, episodic, and hybrid semantic/temporal memory. Test
corrections, conflicts, evidence, consolidation, and deletion across all stores.

In [1]:
from pathlib import Path
import os,subprocess,sys
candidates=[Path.cwd(),Path.cwd()/"project8",Path("/content/ai_agentic_attemptings/project8")]
PROJECT_ROOT=next((p.resolve() for p in candidates if (p/"config/default.json").exists()),None)
if PROJECT_ROOT is None:
    repo=Path("/content/ai_agentic_attemptings")
    if not repo.exists(): subprocess.run(["git","clone","https://github.com/soraber/ai_agentic_attemptings.git",str(repo)],check=True)
    PROJECT_ROOT=repo/"project8"
os.chdir(PROJECT_ROOT)
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    subprocess.run([sys.executable,"-m","pip","install","--upgrade-strategy","only-if-needed","-r","requirements-colab.txt"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-e",".","--no-deps"],check=True)
source_root=PROJECT_ROOT/"src"
if str(source_root) not in sys.path: sys.path.insert(0,str(source_root))
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    check=subprocess.run([sys.executable,"-m","pip","check"],text=True,capture_output=True)
    if check.returncode: print(check.stdout or check.stderr)
from project8_agent.memory import MemoryStore
try: import torch
except ImportError: torch=None
print("Project 8 imports passed")
print({"cuda":bool(torch and torch.cuda.is_available()),"gpu":torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else None})

Project 8 imports passed
{'cuda': False, 'gpu': None}


In [2]:
import getpass,os,sys
from project8_agent.config import load_config
EVAL_BACKEND="openai"  # deterministic | openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"; config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
print(config.model_dump())

{'project_id': 'project8', 'seed': 20260802, 'model': 'gpt-5.6-luna', 'reasoning_effort': 'low', 'working_window_size': 6, 'episodic_top_k': 5, 'qa_per_conversation': 40, 'selected_conversations': 2, 'max_model_calls': 300, 'max_output_tokens': 500, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 1.0, 'output_price_per_million_usd': 6.0, 'local_model': 'Qwen/Qwen2.5-7B-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'local_device': 'cuda', 'local_max_new_tokens': 300}


In [3]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_locomo.py"],check=True)
subset_path=PROJECT_ROOT/"data/cache/locomo_subset.json"
print("LoCoMo subset:",subset_path.relative_to(PROJECT_ROOT).as_posix())

Prepared 2 conversations and 80 QA items
LoCoMo subset: data/cache/locomo_subset.json


In [4]:
import json
subset=json.loads(subset_path.read_text()); lifecycle=json.loads((PROJECT_ROOT/"data/lifecycle_cases.json").read_text())
assert len(subset)==2 and sum(len(item["qa"]) for item in subset)==80
print({"conversations":2,"qa":80,"lifecycle_events":len(lifecycle["events"])})

{'conversations': 2, 'qa': 80, 'lifecycle_events': 7}


In [5]:
from project8_agent.memory import MemoryStore
from project8_agent.schemas import MemoryEvent,MemoryQuery
runtime=PROJECT_ROOT/"output/runtime"; runtime.mkdir(parents=True,exist_ok=True); store=MemoryStore(runtime/"memory.sqlite"); store.reset()
for item in lifecycle["events"]: store.ingest(MemoryEvent.model_validate(item))
query=MemoryQuery.model_validate(lifecycle["queries"][0]); print(store.answer(query,"window").model_dump()); print(store.answer(query,"episodic").model_dump())
embedding_retriever=None
if RUN_LOCAL_GPU_EVAL:
    from project8_agent.local_models import EmbeddingEventRetriever
    from project8_agent.locomo import conversation_events
    embedding_retriever=EmbeddingEventRetriever(config.embedding_model,config.local_device)
    sample_events=conversation_events(subset[0]); print({"dense_retrieval":[item["event_id"] for item in embedding_retriever.retrieve(sample_events,subset[0]["qa"][0]["question"],"hybrid",config.working_window_size,config.episodic_top_k)]})

{'query_id': 'Q01', 'system': 'window', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': False, 'context_tokens': 12, 'latency_ms': 0.13699999544769526}
{'query_id': 'Q01', 'system': 'episodic', 'answer': 'Boston', 'evidence_ids': ['E01'], 'abstained': False, 'conflict': False, 'context_tokens': 2, 'latency_ms': 0.1996249775402248}


In [6]:
for event_id in lifecycle["delete_event_ids"]: store.delete_event(event_id)
correction=MemoryQuery.model_validate(lifecycle["queries"][1]); conflict=MemoryQuery.model_validate(lifecycle["queries"][2])
print(store.answer(correction,"hybrid").model_dump()); print(store.answer(conflict,"hybrid").model_dump()); assert all(store.deletion_verified(e) for e in lifecycle["delete_event_ids"])

{'query_id': 'Q02', 'system': 'hybrid', 'answer': 'green', 'evidence_ids': ['E04'], 'abstained': False, 'conflict': False, 'context_tokens': 4, 'latency_ms': 0.19225000869482756}
{'query_id': 'Q03', 'system': 'hybrid', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': True, 'context_tokens': 4, 'latency_ms': 0.15966698992997408}


In [7]:
import subprocess,sys
result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True); print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 8 tests failed")

............                                                             [100%]
12 passed in 0.07s



In [8]:
from project8_agent.evaluation import evaluate_lifecycle
from project8_agent.locomo import CachedLocalAnswerer, evaluate_locomo
if not RUN_FULL_EVAL: print("Set RUN_FULL_EVAL=True after P08-C07 passes.")
else:
    lifecycle_summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",runtime/"lifecycle_output")
    if RUN_LOCAL_GPU_EVAL:
        result_dir=PROJECT_ROOT/"output/gpu"; local_answerer=CachedLocalAnswerer(config,runtime/"locomo_local_answer_cache.json")
        summary=evaluate_locomo(subset_path,config,runtime/"unused.json",result_dir,lifecycle_summary,answerer=local_answerer,retriever=embedding_retriever,evaluation_mode="locomo_local_gpu")
    elif RUN_API_EVAL:
        summary=evaluate_locomo(subset_path,config,runtime/"locomo_answer_cache.json",PROJECT_ROOT/"output",lifecycle_summary)
    else:
        summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",PROJECT_ROOT/"output")
    print(summary)

{'project': 'Long-Term Memory Agent', 'result_status': 'measured', 'evaluation_mode': 'locomo_api', 'qa_items': 80, 'window': {'exact_match_pct': 0.0, 'mean_token_f1': 0.03200514735466445, 'mean_evidence_recall': 0.0125, 'mean_context_tokens': 90.5}, 'episodic': {'exact_match_pct': 0.0, 'mean_token_f1': 0.11495915520963307, 'mean_evidence_recall': 0.22291666666666665, 'mean_context_tokens': 71.2875}, 'hybrid': {'exact_match_pct': 0.0, 'mean_token_f1': 0.09412888492296416, 'mean_evidence_recall': 0.19166666666666668, 'mean_context_tokens': 69.6375}, 'model': 'gpt-5.6-luna', 'retrieval_model': 'lexical', 'model_calls': 240, 'cache_hits': 240, 'input_tokens': 109387, 'output_tokens': 17407, 'estimated_cost_usd': 0.213829, 'usage_source': 'cumulative measured cache population', 'deletion_compliance_pct': 100.0, 'lifecycle_validation': {'project': 'Long-Term Memory Agent', 'result_status': 'measured', 'window': {'exact_match_pct': 75.0, 'mean_token_f1': 0.75, 'mean_evidence_recall': 0.875, 

In [9]:
import json
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output")
path=result_dir/"project8_representative_samples.json"; print(json.loads(path.read_text()) if path.exists() else "Run P08-C08 first.")

{'best_hybrid': [], 'hybrid_failures': [{'answer': 'Caroline went to the LGBTQ support group on 7 May 2023, the day before she mentioned it on 8 May 2023.', 'context_tokens': 65, 'evidence_ids': ['D1:3'], 'evidence_recall': 1.0, 'exact_match': False, 'gold': '7 May 2023', 'qa_index': 0, 'question': 'When did Caroline go to the LGBTQ support group?', 'sample_id': 'conv-26', 'system': 'hybrid', 'token_f1': 0.25}, {'answer': 'Melanie painted a sunrise last year, as of 8 May 2023.', 'context_tokens': 51, 'evidence_ids': ['D1:14'], 'evidence_recall': 0.0, 'exact_match': False, 'gold': '2022', 'qa_index': 1, 'question': 'When did Melanie paint a sunrise?', 'sample_id': 'conv-26', 'system': 'hybrid', 'token_f1': 0.0}, {'answer': 'The context does not provide enough information to determine which fields Caroline would pursue in her education.', 'context_tokens': 94, 'evidence_ids': [], 'evidence_recall': 0.0, 'exact_match': False, 'gold': 'Psychology, counseling certification', 'qa_index': 2, 

In [10]:
import subprocess,sys
result_dir=PROJECT_ROOT/("output/gpu" if RUN_LOCAL_GPU_EVAL else "output"); summary_path=result_dir/"project8_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project8_report.pdf")],check=True); subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")

Wrote project8_report.pdf
Project 8 structure and privacy checks passed.
